# OWL-ViT Object Localization untuk Pakaian

OWL-ViT (Vision Transformer for Open-World Localization) adalah model dari Google yang dapat mendeteksi objek berdasarkan text prompt tanpa perlu training khusus.

## 1. Install dan Import Library

In [ ]:
# Install library yang diperlukan
!pip install transformers pillow matplotlib pandas

In [ ]:
import torch
import pandas as pd
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from transformers import OwlViTProcessor, OwlViTForObjectDetection
import os

## 2. Load Model OWL-ViT

In [ ]:
# Load model dan processor OWL-ViT
processor = OwlViTProcessor.from_pretrained("google/owlvit-base-patch32")
model = OwlViTForObjectDetection.from_pretrained("google/owlvit-base-patch32")

print("Model OWL-ViT berhasil dimuat!")

## 3. Load Data dari train.csv

In [ ]:
# Load dataset
df = pd.read_csv('train.csv')
print(f"Total data: {len(df)}")
print("\nSample data:")
print(df.head())

# Path ke folder gambar
train_folder = 'train/train/'

## 4. Fungsi untuk Object Localization

Fungsi ini akan:
1. Load image
2. Menggunakan text prompts untuk mencari objek pakaian
3. Mengembalikan bounding boxes dari objek yang terdeteksi

In [ ]:
def detect_clothing_with_owlvit(image_path, text_queries, threshold=0.1):
    """
    Deteksi pakaian menggunakan OWL-ViT
    
    Args:
        image_path: Path ke gambar
        text_queries: List text prompts untuk objek yang ingin dideteksi
        threshold: Confidence threshold (default 0.1)
    
    Returns:
        image: PIL Image dengan bounding boxes
        results: Dictionary berisi deteksi
    """
    # Load image
    image = Image.open(image_path)
    
    # Process dengan OWL-ViT
    inputs = processor(text=text_queries, images=image, return_tensors="pt")
    
    # Inference
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Post-processing: ambil bounding boxes dengan confidence > threshold
    target_sizes = torch.Tensor([image.size[::-1]])
    results = processor.post_process_object_detection(
        outputs=outputs, 
        target_sizes=target_sizes, 
        threshold=threshold
    )[0]
    
    # Gambar bounding boxes
    draw = ImageDraw.Draw(image)
    
    detection_results = []
    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        box = [round(i, 2) for i in box.tolist()]
        label_text = text_queries[label]
        confidence = round(score.item(), 3)
        
        # Gambar rectangle
        draw.rectangle(box, outline="red", width=3)
        draw.text((box[0], box[1] - 10), f"{label_text}: {confidence}", fill="red")
        
        detection_results.append({
            "label": label_text,
            "confidence": confidence,
            "bbox": box
        })
        
        print(f"Terdeteksi: {label_text} dengan confidence {confidence} di {box}")
    
    return image, detection_results

## 5. Contoh Penggunaan - Deteksi Pakaian

Kita akan mencoba mendeteksi berbagai jenis pakaian dengan text prompts.

In [ ]:
# Ambil sample gambar pertama
sample_id = df.iloc[0]['id']
image_path = os.path.join(train_folder, f'{sample_id}.jpg')

# Text queries untuk berbagai jenis pakaian
text_queries = [
    "shirt",
    "t-shirt", 
    "dress",
    "pants",
    "jacket",
    "sweater",
    "skirt",
    "clothing",
    "garment"
]

print(f"Memproses gambar: {image_path}")
print(f"Text queries: {text_queries}\n")

# Deteksi pakaian
result_image, detections = detect_clothing_with_owlvit(image_path, text_queries, threshold=0.1)

# Tampilkan hasil
plt.figure(figsize=(12, 8))
plt.imshow(result_image)
plt.axis('off')
plt.title(f'Hasil Deteksi Pakaian - Image ID: {sample_id}')
plt.tight_layout()
plt.show()

print(f"\nTotal objek terdeteksi: {len(detections)}")

## 6. Deteksi Multiple Images

Contoh untuk memproses beberapa gambar sekaligus.

In [ ]:
# Proses 4 gambar pertama
num_samples = 4
sample_data = df.head(num_samples)

fig, axes = plt.subplots(2, 2, figsize=(15, 15))
axes = axes.flatten()

for idx, (_, row) in enumerate(sample_data.iterrows()):
    image_id = row['id']
    image_path = os.path.join(train_folder, f'{image_id}.jpg')
    
    print(f"\n{'='*50}")
    print(f"Processing Image ID: {image_id}")
    print(f"{'='*50}")
    
    # Deteksi dengan threshold lebih tinggi untuk hasil yang lebih akurat
    result_image, detections = detect_clothing_with_owlvit(
        image_path, 
        text_queries, 
        threshold=0.15
    )
    
    # Tampilkan di subplot
    axes[idx].imshow(result_image)
    axes[idx].axis('off')
    axes[idx].set_title(f'ID: {image_id} - {len(detections)} deteksi')

plt.tight_layout()
plt.show()

## 8. Contoh dengan Text Prompts yang Lebih Spesifik

Anda bisa menggunakan text prompts yang lebih spesifik untuk deteksi yang lebih akurat.

In [ ]:
# Text prompts yang lebih spesifik berdasarkan warna dan jenis
specific_queries = [
    "red shirt",
    "blue dress", 
    "black pants",
    "white t-shirt",
    "green jacket",
    "striped shirt",
    "casual wear",
    "formal clothing"
]

# Coba dengan gambar sample
sample_id = df.iloc[5]['id']  # Ambil gambar ke-6
image_path = os.path.join(train_folder, f'{sample_id}.jpg')

print(f"Memproses dengan text prompts spesifik:")
print(specific_queries)
print(f"\nImage ID: {sample_id}\n")

result_image, detections = detect_clothing_with_owlvit(
    image_path, 
    specific_queries, 
    threshold=0.1
)

plt.figure(figsize=(12, 8))
plt.imshow(result_image)
plt.axis('off')
plt.title(f'Deteksi dengan Prompts Spesifik - Image ID: {sample_id}')
plt.tight_layout()
plt.show()